In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [2]:
X = tf.range(10)

In [3]:
X

<tf.Tensor: shape=(10,), dtype=int32, numpy=array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)>

In [4]:
dataset = tf.data.Dataset.from_tensor_slices(X)

In [5]:
for item in dataset:
    print(item)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)


In [6]:
dataset = dataset.repeat(3).batch(7)
for item in dataset:
    print(item)

tf.Tensor([0 1 2 3 4 5 6], shape=(7,), dtype=int32)
tf.Tensor([7 8 9 0 1 2 3], shape=(7,), dtype=int32)
tf.Tensor([4 5 6 7 8 9 0], shape=(7,), dtype=int32)
tf.Tensor([1 2 3 4 5 6 7], shape=(7,), dtype=int32)
tf.Tensor([8 9], shape=(2,), dtype=int32)


In [7]:
dataset = dataset.map(lambda x:x*2)
for i in dataset:
    print(i)

tf.Tensor([ 0  2  4  6  8 10 12], shape=(7,), dtype=int32)
tf.Tensor([14 16 18  0  2  4  6], shape=(7,), dtype=int32)
tf.Tensor([ 8 10 12 14 16 18  0], shape=(7,), dtype=int32)
tf.Tensor([ 2  4  6  8 10 12 14], shape=(7,), dtype=int32)
tf.Tensor([16 18], shape=(2,), dtype=int32)


In [8]:
dataset = dataset.apply(tf.data.experimental.unbatch())

Instructions for updating:
Use `tf.data.Dataset.unbatch()`.


In [9]:
new_dataset = tf.range(10)

In [10]:
new_dataset = tf.data.Dataset.from_tensor_slices(new_dataset)

In [11]:
new_dataset = new_dataset.filter(lambda x:x<5)

In [12]:
for i in new_dataset:
    print(i)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)


In [13]:
for i in new_dataset.take(1):
    print(i)

tf.Tensor(0, shape=(), dtype=int32)


In [14]:
n_ds = tf.data.Dataset.range(10).repeat(3)

In [15]:
n_ds = n_ds.shuffle(buffer_size=5, seed=42).batch(7)

In [16]:
for i in n_ds:
    print(i)

tf.Tensor([0 2 3 6 7 9 4], shape=(7,), dtype=int64)
tf.Tensor([5 0 1 1 8 6 5], shape=(7,), dtype=int64)
tf.Tensor([4 8 7 1 2 3 0], shape=(7,), dtype=int64)
tf.Tensor([5 4 2 7 8 9 9], shape=(7,), dtype=int64)
tf.Tensor([3 6], shape=(2,), dtype=int64)


In [17]:
# filepath_ds = tf.data.Dataset.list_files(train_filepaths, seed=42)
# n_readers = 5 
# dataset = filepath_dataset.interleave(
#     lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
#     cycle_length=n_readers)

In [18]:
# for line in dataset.take(5):
#     print(line.numpy())

In [19]:
# X_mean, X_std = []
# n_inputs = 8

# def preprocess(line):
#     defs = [0.] * n_inputs + [tf.constant([]. dtype=tf.float32)]
#     fields = tf.io.decode_csv(line, record_defaults=def)
#     x = tf.stack(fields[:-1])
#     y = tf.stack(fields[-1:])
#     return(x-X_mean) / X_std, y

In [20]:
def csv_reader_dataset(filepaths, repeat=1, n_readers=5,
                       n_read_threads=None, shuffle_buffer_size=10000,
                       n_parse_threads=5, batch_size=32):
    dataset = tf.data.Dataset.list_files(filepaths).repeat(repeat)
    dataset = dataset.interleave(
        lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
        cycle_length=n_readers, num_parallel_calls=n_read_threads)
    dataset = dataset.shuffle(shuffle_buffer_size)
    dataset = dataset.map(preprocess, num_parallel_calls=n_parse_threads)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(1)

In [21]:
# train_set = csv_reader_dataset(train_filepaths)
# valid_set = csv_reader_dataset(valid_filepaths)
# test_set = csv_reader_dataset(test_filepaths)

#model = Sequential([..])
#model.compile()
#model.fit(train_set,epochs=100,validation_data=valid_set)

In [22]:
with tf.io.TFRecordWriter("my_data.tfrecord") as f:
    f.write(b"this is the first record")
    f.write(b"and this is the second record")

In [23]:
filepaths = ["my_data.tfrecord"]
dataset = tf.data.TFRecordDataset(filepaths)
for item in dataset:
    print(item)

tf.Tensor(b'this is the first record', shape=(), dtype=string)
tf.Tensor(b'and this is the second record', shape=(), dtype=string)


In [24]:
options = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter("my_compressed.tfrecord",options) as f:
    f.write(b"this is the compressed record")
    f.write(b"and this is the compressed second record")

In [25]:
dataset = tf.data.TFRecordDataset(["my_compressed.tfrecord"],
                                  compression_type="GZIP")

In [26]:
# syntax = "proto3";
# message Person{
#     string name = 1;
#     int32 id =2;
#     repeated string email = 3;}

In [27]:
# from person_pb2 import Person

# person = Person(name="AL", id=123, email=["a@b.com"])

In [28]:
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

In [29]:
person_example = Example(
    features = Features(
        feature={
            "name": Feature(bytes_list = BytesList(value=[b"Alice"])),
            "id": Feature(int64_list = Int64List(value=[123])),
            "emails": Feature(bytes_list = BytesList(value=[b"a@b.com", b"c@d.com"]))
        }))

In [30]:
with tf.io.TFRecordWriter("my_contacts.tfrecord") as f:
    f.write(person_example.SerializeToString())

In [31]:
feature_description = {
    "name" : tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "emails":tf.io.VarLenFeature(tf.string),
}

for serialised_example in tf.data.TFRecordDataset(["my_contacts.tfrecord"]):
    parsed_example = tf.io.parse_single_example(serialised_example, feature_description)

In [32]:
tf.sparse.to_dense(parsed_example["emails"], default_value=b"")

<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [33]:
parsed_example["emails"].values

<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [34]:
dataset = tf.data.TFRecordDataset(["my_contacts.tfrecord"]).batch(10)
for serialised_examples in dataset:
    parsed_examples = tf.io.parse_example(serialised_examples, feature_description)

In [35]:
# means = np.mean(X_train, axis=0, keepdims=True)
# stds = np.std(X_train,axis=0, keepdims=True)
# eps = keras.backend.epsilon()
# model = keras.models.Sequential([
#     keras.layers.Lambda(lambda inputs: (inputs - means) / (stds + eps)),
#     [...] #other layers
# ])

In [36]:
class Standardization(keras.layers.Layer):
    def adapt(self, data_sample):
        self.means_ = np.mean(data_sample, axis=0, keepdims=True)
        self.stds_ = np.std(data_sample, axis=0, keepdims=True)
    def call(self, inputs):
        return (inputs - self.means_) / (self.stds_ + keras.backend.epsilon())

In [37]:
# std_layer = Standardization()
# std_layer.adapt(data_sample)

In [38]:
vocab = ["<1H OCEAN","INLAND","NEAR OCEAN","NEAR BAY", "ISLAND"]
indices = tf.range(len(vocab), dtype=tf.int64)

In [39]:
indices

<tf.Tensor: shape=(5,), dtype=int64, numpy=array([0, 1, 2, 3, 4])>

In [40]:
table_init = tf.lookup.KeyValueTensorInitializer(vocab, indices)
num_oov_buckets = 2
table = tf.lookup.StaticVocabularyTable(table_init, num_oov_buckets)

In [41]:
table

In [42]:
categories = tf.constant(["NEAR BAY", "DESERT", "INLAND"])
cat_indices = table.lookup(categories)

In [43]:
cat_one_hot = tf.one_hot(cat_indices, depth=len(vocab) + num_oov_buckets)

In [44]:
cat_one_hot

<tf.Tensor: shape=(3, 7), dtype=float32, numpy=
array([[0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 0., 0.]], dtype=float32)>

In [45]:
embedding_dim = 2 
embed_init = tf.random.uniform([len(vocab) + num_oov_buckets, embedding_dim])

In [46]:
embed_init

<tf.Tensor: shape=(7, 2), dtype=float32, numpy=
array([[0.3782128 , 0.14365828],
       [0.8683208 , 0.15964079],
       [0.3125509 , 0.7733778 ],
       [0.5589129 , 0.8174505 ],
       [0.76790917, 0.43997967],
       [0.5991895 , 0.5253434 ],
       [0.51228666, 0.13880908]], dtype=float32)>

In [47]:
embedding_matrix = tf.Variable(embed_init)

In [48]:
embedding_matrix

<tf.Variable 'Variable:0' shape=(7, 2) dtype=float32, numpy=
array([[0.3782128 , 0.14365828],
       [0.8683208 , 0.15964079],
       [0.3125509 , 0.7733778 ],
       [0.5589129 , 0.8174505 ],
       [0.76790917, 0.43997967],
       [0.5991895 , 0.5253434 ],
       [0.51228666, 0.13880908]], dtype=float32)>

In [49]:
tf.nn.embedding_lookup(embedding_matrix, cat_indices)

<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
array([[0.5589129 , 0.8174505 ],
       [0.5991895 , 0.5253434 ],
       [0.8683208 , 0.15964079]], dtype=float32)>

In [50]:
embedding = keras.layers.Embedding(input_dim=len(vocab) + num_oov_buckets, output_dim=embedding_dim)

In [51]:
embedding

<Embedding name=embedding, built=False>

In [52]:
embedding(cat_indices)

<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
array([[-0.03960717,  0.0471054 ],
       [ 0.00965561,  0.03790117],
       [-0.01252489, -0.04579996]], dtype=float32)>

In [53]:
reg_inputs = keras.layers.Input(shape=[8])
categories = keras.layers.Input(shape=[], dtype=tf.string)
cat_indices = keras.layers.Lambda(
    lambda cats: table.lookup(cats), output_shape=()
)(categories)
cat_embed = keras.layers.Embedding(input_dim=6, output_dim=2)(cat_indices)
encoded_inputs = keras.layers.concatenate([reg_inputs, cat_embed])
outputs = keras.layers.Dense(1)(encoded_inputs)
model = keras.models.Model(inputs=[reg_inputs,categories],outputs=[outputs])

In [54]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None)                    │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lambda (Lambda)               │ (None)                    │               0 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer (InputLayer)      │ (None, 8)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_1 (Embedding)       │ (None, 2)                 │              12 │ lambda[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 10)                │               0 │ input_layer[0][0],         │
│                               │                           │                 │ embedding_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 1)                 │              11 │ concatenate[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 23 (92.00 B)

 Trainable params: 23 (92.00 B)

 Non-trainable params: 0 (0.00 B)

In [55]:
# normalization = keras.layers.Normalization()
# discreatization = keras.layers.Discretization([...])
# pipline = keras.layers.PreprocessingStage([normalization,discretization])
# pipeline.adapt(data_sample)

In [56]:
# import tensorflow_transform as tft

# def preprocess(inputs):
#     median_age = inputs["housing_median_age"]
#     ocean_proximity = inputs["ocean_proximity"]
#     standarized_age = tft.scale_to_z_score(median_age)
#     ocean_proximity_id = tf.compute_and_apply_vocabulary(ocean_promixity)
#     return {
#     "standardized_median_age":standardized_age,
#     "ocean_proximity_id": ocean_proximity_id
#     }

In [57]:
import tensorflow_datasets as tfds

dataset = tfds.load(name="mnist")

In [58]:
mnist_train, mnist_test = dataset["train"], dataset["test"]

In [59]:
mnist_train = mnist_train.shuffle(10000).batch(32).prefetch(1)

In [60]:
for item in mnist_train:
    images = item["image"]
    labels = item["label"]

In [61]:
mnist_train = mnist_train.shuffle(10000).batch(32)
mnist_train = mnist_train.map(lambda items: (items["image"], items["label"]))
mnist_train = mnist_train.prefetch(1)

In [62]:
# dataset = tfds.load(name="mnist",batch_size=32, as_supervised=True)
# mnist_train = dataset["train"].prefetch(1)
# model = keras.models.Sequential([...])
# model.compile(loss="...",optimizer="sgd")
# model.fit(mnist_train,epochs=5)

In [64]:
mnist_train

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None, 28, 28, 1), dtype=tf.uint8, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

In [65]:
##exercises

In [94]:
import tensorflow as tf
import tensorflow_datasets as tfds

train_ds, valid_ds, test_ds = tfds.load(
    "mnist", split=["train[:90%]", "train[90%:]", "test"], as_supervised=True)

def serialise(image, label):
    image_bytes = tf.io.serialize_tensor(image).numpy()
    example = tf.train.Example(features=tf.train.Features(feature={
        "image": tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_bytes])),
        "label": tf.train.Feature(int64_list=tf.train.Int64List(value=[label.numpy()])),
    }))
    return example.SerializeToString()

def write_shards(dataset, name, n_shards=10):
    paths = [f"{name}-{i:03d}-of-{n_shards:03d}.tfrecord" for i in range(n_shards)]
    writers = [tf.io.TFRecordWriter(p) for p in paths]
    for i, (image, label) in enumerate(dataset):
        writers[i % n_shards].write(serialise(image, label))
    for w in writers:
        w.close()
    return paths

train_paths = write_shards(train_ds, "mnist_train", 10)
valid_paths = write_shards(valid_ds, "mnist_valid", 2)
test_paths  = write_shards(test_ds,  "mnist_test",  2)

In [98]:
feature_description = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

def parse(serialised):
    example = tf.io.parse_single_example(serialised, feature_description)
    image = tf.io.parse_tensor(example["image"], out_type=tf.uint8)
    image = tf.reshape(image, [28, 28, 1])
    return tf.cast(image, tf.float32) / 255.0, example["label"]

def read_shards(paths, shuffle=False, batch_size=32, cache=True):
    ds = tf.data.Dataset.list_files(paths, shuffle=shuffle)
    ds = ds.interleave(tf.data.TFRecordDataset,
                       cycle_length=5, num_parallel_calls=tf.data.AUTOTUNE)
    if cache:
        ds = ds.cache()
    ds = ds.map(parse, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(10000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_set = read_shards(train_paths, shuffle=True)
valid_set = read_shards(valid_paths)
test_set  = read_shards(test_paths)

In [99]:
class Standardisation(tf.keras.layers.Layer):
    def adapt(self, data_sample):
        self.means_ = tf.reduce_mean(data_sample, axis=0, keepdims=True)
        self.stds_ = tf.math.reduce_std(data_sample, axis=0, keepdims=True)
    def call(self, inputs):
        return (inputs - self.means_) / (self.stds_ + tf.keras.backend.epsilon())

standardisation = Standardisation(input_shape=[28, 28, 1])
sample = tf.concat([images for images, labels in train_set.take(100)], axis=0)
standardisation.adapt(sample)

model = tf.keras.Sequential([
    standardisation,
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])

C:\Users\Berkay\AppData\Local\Temp\ipykernel_33440\2197757162.py:8: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  standardisation = Standardisation(input_shape=[28, 28, 1])


In [100]:
tb = tf.keras.callbacks.TensorBoard(log_dir="logs", profile_batch=(100, 200))
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="nadam", metrics=["accuracy"])
model.fit(train_set, validation_data=valid_set, epochs=5, callbacks=[tb])

Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9222 - loss: 660.0674 - val_accuracy: 0.9558 - val_loss: 394.1220
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9602 - loss: 445.4828 - val_accuracy: 0.9613 - val_loss: 956.8955
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9695 - loss: 450.4534 - val_accuracy: 0.9637 - val_loss: 463.4295
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9737 - loss: 325.9438 - val_accuracy: 0.9628 - val_loss: 359.7467
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9791 - loss: 217.5399 - val_accuracy: 0.9628 - val_loss: 377.5675


In [102]:
%load_ext tensorboard
%tensorboard --logdir=logs